## Libraries

In [ ]:
!pip install torch_geometric
!pip install torch-spline-conv


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os.path as osp
import torch
import torch.nn.functional as F
import torch.nn as nn
import time
import copy

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SplineConv
from torch_geometric.typing import WITH_TORCH_SPLINE_CONV

import torch.nn.utils.prune as prune
from torch.nn.utils.prune import global_unstructured, L1Unstructured
import statistics as stat

In [2]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=True) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

  

## Loading Dataset

In [3]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [4]:

if not WITH_TORCH_SPLINE_CONV:
    quit("This example requires 'torch-spline-conv'")

dataset = 'Cora'
transform = T.Compose([
    T.RandomNodeSplit(num_val=500, num_test=500),
    T.TargetIndegree(),
])
path =  'data'
dataset = Planetoid(path, dataset, transform=transform)
data = dataset[0]

## Model

In [5]:


class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SplineConv(dataset.num_features, 16, dim=1, kernel_size=2)
        self.conv2 = SplineConv(16, dataset.num_classes, dim=1, kernel_size=2)

    def forward(self):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.dropout(x, training=self.training)
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index, edge_attr)
        return F.log_softmax(x, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

In [6]:


def train():
    model.train()
    optimizer.zero_grad()
    F.nll_loss(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()
    


@torch.no_grad()
def test(model):
    model.eval()
    log_probs, accs = model(), []
    for _, mask in data('train_mask', 'test_mask'):
        pred = log_probs[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs




In [7]:


for epoch in range(10):
    train()
    train_acc, test_acc = test(model)
    #if epoch % 10 == 0:
    print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

Epoch: 000, Train: 0.4889, Test: 0.4780
Epoch: 001, Train: 0.4854, Test: 0.4680
Epoch: 002, Train: 0.5486, Test: 0.4920
Epoch: 003, Train: 0.6552, Test: 0.5840
Epoch: 004, Train: 0.7641, Test: 0.7120
Epoch: 005, Train: 0.8495, Test: 0.7920
Epoch: 006, Train: 0.8958, Test: 0.8460
Epoch: 007, Train: 0.9069, Test: 0.8620
Epoch: 008, Train: 0.9110, Test: 0.8560
Epoch: 009, Train: 0.9128, Test: 0.8600


In [8]:
best_checkpoint = dict()
best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
model.load_state_dict(best_checkpoint['state_dict'])
recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

In [9]:
t0=time.time()
train_acc, base_model_accuracy=test(model)
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model, count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"base model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"base model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")

base model has accuracy on test set=0.86%
base model has size=0.26 MiB
The time inference of base model is =16.691665410995483
The number of parametrs of base model is:69143


pseudo quantization
---

The following code is for pseudo quantization.Pseudo Quantization is used to simulate the effects of quantization on a model without actually quantizing the model's weights. (i.e. rounding to the nearest quantized value and then dequantizing back to a float.)

In [10]:
model

Net(
  (conv1): SplineConv(1433, 16, dim=1)
  (conv2): SplineConv(16, 7, dim=1)
)

In [11]:
for name, param in model.named_parameters():
  
        print(name)

conv1.weight
conv1.bias
conv1.lin.weight
conv2.weight
conv2.bias
conv2.lin.weight


In [7]:
# core quantization method (simulated quantization)
def pseudo_quantize_tensor(w, n_bit=4, q_group_size=-1):
    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)

    assert w.dim() == 2

    # Calculate the maximum (\alpha) and minimum values (\beta) in the tensor.
    max_val = w.amax(dim=1, keepdim=True)
    assert max_val.dim() == 2 and max_val.size(0) == w.size(0) and max_val.size(1) == 1
    min_val = w.amin(dim=1, keepdim=True)
    assert min_val.dim() == 2 and min_val.size(0) == w.size(0) and min_val.size(1) == 1

    # Calculate the scale factor and zero point.  (Formula 1 & 2)
    max_int = 2 ** n_bit - 1
    scales = (max_val - min_val).clamp(min=1e-5) / max_int
    assert scales.shape == max_val.shape
    zeros = (-torch.round(min_val / scales)).clamp_(0, max_int)
    assert scales.shape == min_val.shape

    assert torch.isnan(scales).sum() == 0
    assert torch.isnan(w).sum() == 0

    # Quantize W: Map values in the range [\beta, \alpha] to lie within [0, 2^b - 1] (Formula 3)
    w = torch.clamp(torch.round(w / scales) + zeros, 0, max_int)
    assert w.dim() == 2 and w.size(0) == scales.size(0) and w.size(1) == q_group_size

    # Dequantize W (pseudo quantization, the inverse transformation of Formula 3)
    w = (w - zeros) * scales
    assert w.dim() == 2 and w.size(0) == scales.size(0) and w.size(1) == q_group_size

    assert torch.isnan(w).sum() == 0

    w = w.reshape(org_w_shape)
    return w

@torch.no_grad()
def pseudo_quantize_model_weight(model, w_bit):
    for n, m in model.named_parameters():
        #if isinstance(m, nn.Linear):
        q_group_size=m.data.shape[-1]
        if 'lin.weight' in n:
            m.data = pseudo_quantize_tensor(m.data, n_bit=w_bit, q_group_size=q_group_size)
            #print(m.data)

In [18]:
pseudo_quantize_model_weight(model, w_bit=3)

t0=time.time()
awq_train_acc,awq_model_accuracy=test(model)
t1=time.time()
t_awq_model=t1 - t0
###
awq_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_awq_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"awq model has accuracy on test set={awq_model_accuracy:.2f}%")
print(f"awq model has size={awq_model_size/MiB:.2f} MiB")
print(f"The time inference of awq model is ={t_awq_model}") 
print(f"The number of parametrs of awq model is:{num_parm_awq_model}")
print(f"awq model has size={awq_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/awq_model_size:.2f} X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")

0.858
awq model has accuracy on test set=0.86%
awq model has size=0.24 MiB
The time inference of awq model is =17.993947505950928
The number of parametrs of awq model is:63565
awq model has size=0.24 MiB, which is 1.09 X smaller than the 0.26 MiB dense model


## Finetuning Q-Model

In [ ]:

best_sparse_checkpoint = dict()
best_sparse_accuracy = 0
num_finetune_epochs=5
print(f'Finetuning Fine-grained Pruned Sparse Model')
for epoch in range(num_finetune_epochs):
    # At the end of each train iteration, we have to apply the pruning mask
    #    to keep the model sparse during the training
    train()
    train_acc, accuracy = test(model)
  
    is_best = accuracy > best_sparse_accuracy
    if is_best:
        best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        best_sparse_accuracy = accuracy
    
    #if epoch % 10 == 0:
    print(f'Epoch {epoch} Accuracy {accuracy:.2f}% / Best Sparse Accuracy: {best_sparse_accuracy:.2f}%')

            
model.load_state_dict(best_sparse_checkpoint['state_dict'])

t0=time.time()
train_acc, pruned_finetune_model_accuracy=test(model)
t1=time.time()
t_pruned_finetune_model=t1 - t0
###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")            

## Manual Measurement

In [8]:
import statistics as stat


Eva_final=dict()

Base_model_accuracy=[]
T_base_model=[]
Num_parm_base_model=[]
Base_model_size=[]

AWQ_model_accuracy=[]
T_AWQ_model=[]
Num_parm_AWQ_model=[]
AWQ_model_size=[]

AWQ_finetune_model_accuracy=[]
T_AWQ_finetune_model=[]
Num_parm_AWQ_finetune_model=[]
AWQ_finetune_model_size=[]


In [9]:
num_epoch=50
for i in range(3):

        print(f'This is the iteration {i}')
        Eva=dict()

        print(f'Training and evaluation before quatization ')
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model, data = Net().to(device), data.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

        for epoch in range(num_epoch):
            train()
            train_acc, test_acc = test(model)
            if epoch % 10 == 0:
                print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

        # report test msg

        t0=time.time()
        train_acc, base_model_accuracy=test(model)
        t1=time.time()
        t_base_model=t1 - t0
        ###
        base_model_size = get_model_size(model,count_nonzero_only=True)
        num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
        ###   
        print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
        print(f"dense model has size={base_model_size/MiB:.2f} MiB")
        print(f"The time inference of base model is ={t_base_model}") 
        print(f"The number of parametrs of base model is:{num_parm_base_model}")    

        #Update my Eva dictionary
        Eva.update({'base model accuracy': base_model_accuracy,
                    'time inference of base model': t_base_model,
                    'number parmameters of base model': num_parm_base_model,
                    'size of base model': base_model_size})

        print('_______________________________________________________')
        print('Quatization')
        pseudo_quantize_model_weight(model, w_bit=3)

        t0=time.time()
        train_acc, awq_model_accuracy=test(model)
        t1=time.time()
        t_awq_model=t1 - t0
        ###
        awq_model_size = get_model_size(model,count_nonzero_only=True)
        num_parm_awq_model=get_num_parameters(model, count_nonzero_only=True)
        ###   
        print(f"awq model has accuracy on test set={awq_model_accuracy:.2f}%")
        print(f"awq model has size={awq_model_size/MiB:.2f} MiB")
        print(f"The time inference of awq model is ={t_awq_model}") 
        print(f"The number of parametrs of awq model is:{num_parm_awq_model}")
        print(f"awq model has size={awq_model_size/MiB:.2f} MiB, "
              f"which is {base_model_size/awq_model_size:.2f} X smaller than "
              f"the {base_model_size/MiB:.2f} MiB base model")

        #Update my Eva dictionary
        Eva.update({'awq model accuracy': awq_model_accuracy,
                    'time inference of awq model': t_awq_model,
                    'number parmameters of awq model': num_parm_awq_model,
                    'size of awq model': awq_model_size})

        print('_______________________________________________________')
        print(f'Finetuning Q Model')
        for epoch in range(num_epoch):
            train()
            train_acc, test_acc = test(model)
            if epoch % 10 == 0:
                print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')



        t0=time.time()
        train_acc, awq_finetune_model_accuracy=test(model)
        t1=time.time()
        t_awq_finetune_model=t1 - t0
        ###
        awq_finetune_model_size = get_model_size(model,count_nonzero_only=True)
        num_parm_awq_finetune_model=get_num_parameters(model, count_nonzero_only=True)
        ###   
        print(f"awq model has accuracy on test set={awq_finetune_model_accuracy:.2f}%")
        print(f"awq model has size={awq_finetune_model_size/MiB:.2f} MiB")
        print(f"The time inference of awq model is ={t_awq_finetune_model}") 
        print(f"The number of parametrs of awq model is:{num_parm_awq_finetune_model}")
        print(f"awq model has size={awq_finetune_model_size/MiB:.2f} MiB, "
        f" which is {base_model_size/awq_finetune_model_size:.2f} X smaller than "
        f" the {base_model_size/MiB:.2f} MiB dense model")   
        #Update my Eva dictionary
        Eva.update({'awq and finetune model accuracy': awq_finetune_model_accuracy,
                    'time inference of awq and finetune model': t_awq_finetune_model,
                    'number parmameters of awq and finetune model': num_parm_awq_finetune_model,
                    'size of awq and finetune model':  awq_finetune_model_size})





        Base_model_accuracy.append(Eva['base model accuracy'])
        T_base_model.append(Eva['time inference of base model'])
        Num_parm_base_model.append(int(Eva['number parmameters of base model']))
        Base_model_size.append(int(Eva['size of base model']))

        AWQ_model_accuracy.append(Eva['awq model accuracy'])
        T_AWQ_model.append(Eva['time inference of awq model'])
        Num_parm_AWQ_model.append(int(Eva['number parmameters of awq model']))
        AWQ_model_size.append(int(Eva['size of awq model']))

        AWQ_finetune_model_accuracy.append(Eva['awq and finetune model accuracy'])
        T_AWQ_finetune_model.append(Eva['time inference of awq and finetune model'])
        Num_parm_AWQ_finetune_model.append(int(Eva['number parmameters of awq and finetune model']))
        AWQ_finetune_model_size.append(int(Eva['size of awq and finetune model']))




This is the iteration 0
Training and evaluation before quatization 
Epoch: 000, Train: 0.2974, Test: 0.2920
Epoch: 010, Train: 0.9087, Test: 0.8580
Epoch: 020, Train: 0.9584, Test: 0.9080
Epoch: 030, Train: 0.9696, Test: 0.9120
Epoch: 040, Train: 0.9719, Test: 0.8960
dense model has accuracy on test set=0.91%
dense model has size=0.26 MiB
The time inference of base model is =16.270809173583984
The number of parametrs of base model is:69143
_______________________________________________________
Quatization
awq model has accuracy on test set=0.91%
awq model has size=0.21 MiB
The time inference of awq model is =16.360447645187378
The number of parametrs of awq model is:55912
awq model has size=0.21 MiB, which is 1.24 X smaller than the 0.26 MiB base model
_______________________________________________________
Finetuning Q Model
Epoch: 000, Train: 0.9666, Test: 0.9080
Epoch: 010, Train: 0.9731, Test: 0.9120
Epoch: 020, Train: 0.9760, Test: 0.9000
Epoch: 030, Train: 0.9789, Test: 0.8960
E

In [10]:
Eva_final=dict()
base_model_accuracy_mean = stat.mean(Base_model_accuracy)
base_model_accuracy_std =  stat.stdev(Base_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)

Eva_final.update({'base model accuracy':float(format(base_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of base model accuracy':float(format(base_model_accuracy_std, '.3f'))})
                 
t_base_model_mean =stat.mean(T_base_model)
t_base_model_std =stat.stdev(T_base_model)  
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of base model':float(format(t_base_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of base model':float(format(t_base_model_std, '.3f'))})


num_parm_base_model_mean = stat.mean(Num_parm_base_model)
num_parm_base_model_std = stat.stdev(Num_parm_base_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of base model':num_parm_base_model_mean})
Eva_final.update({'Std of number parmameters of base model':num_parm_base_model_std})

base_model_size_mean = stat.mean(Base_model_size)
base_model_size_std = stat.stdev(Base_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'base_model_size':base_model_size_mean})
Eva_final.update({'Std of base_model_size':base_model_size_std})

#################################


AWQ_model_accuracy_mean =stat.mean(AWQ_model_accuracy)
AWQ_model_accuracy_std = stat.stdev(AWQ_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ model accuracy':float(format(AWQ_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of AWQ model accuracy':float(format(AWQ_model_accuracy_std, '.3f'))})
                 

t_AWQ_model_mean = stat.mean(T_AWQ_model)
t_AWQ_model_std =stat.stdev(T_AWQ_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of AWQ model':float(format(t_AWQ_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of AWQ model':float(format(t_AWQ_model_std, '.3f'))})

num_parm_AWQ_model_mean = stat.mean(Num_parm_AWQ_model)
num_parm_AWQ_model_std = stat.stdev(Num_parm_AWQ_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of AWQ model':num_parm_AWQ_model_mean})
Eva_final.update({'Std of number parmameters of AWQ model':num_parm_AWQ_model_std})

AWQ_model_size_mean =stat.mean( AWQ_model_size)
AWQ_model_size_std = stat.stdev(AWQ_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ model size':AWQ_model_size_mean})
Eva_final.update({'Std of AWQ_model_size':AWQ_model_size_std })


#################################
#################################
AWQ_finetune_model_accuracy_mean =stat.mean(AWQ_finetune_model_accuracy)
AWQ_finetune_model_accuracy_std = stat.stdev(AWQ_finetune_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ finetune model accuracy':float(format(AWQ_finetune_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of AWQ finetune model accuracy':float(format(AWQ_finetune_model_accuracy_std, '.3f'))})                 

t_AWQ_finetune_model_mean =stat.mean(T_AWQ_finetune_model)
t_AWQ_finetune_model_std =stat.stdev(T_AWQ_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of AWQ finetune model':float(format(t_AWQ_finetune_model_mean,'.3f'))})
Eva_final.update({'Std of time inference of AWQ finetune model':float(format(t_AWQ_finetune_model_std,'.3f'))})

num_parm_AWQ_finetune_model_mean =stat.mean(Num_parm_AWQ_finetune_model)
num_parm_AWQ_finetune_model_std = stat.stdev(Num_parm_AWQ_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of AWQ finetune model':num_parm_AWQ_finetune_model_mean})
Eva_final.update({'Std of number parmameters of AWQ finetune model':num_parm_AWQ_finetune_model_std })

AWQ_finetune_model_size_mean = stat.mean(AWQ_finetune_model_size)
AWQ_finetune_model_size_std = stat.stdev(AWQ_finetune_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ finetune model size':AWQ_finetune_model_size_mean})
Eva_final.update({'Std of AWQ finetune model size':AWQ_finetune_model_size_std})



print(f"All measurement about AWQ Quatization process ")   
Eva_final

All measurement about Quatization process 


{'base model accuracy': 0.905,
 'Std of base model accuracy': 0.01,
 'time inference of base model': 16.252,
 'Std of time inference of base model': 0.042,
 'number parmameters of base model': 69143,
 'Std of number parmameters of base model': 0.0,
 'base_model_size': 2212576,
 'Std of base_model_size': 0.0,
 'AWQ model accuracy': 0.904,
 'Std of AWQ model accuracy': 0.012,
 'time inference of AWQ model': 16.298,
 'Std of time inference of AWQ model': 0.056,
 'number parmameters of AWQ model': 55657.666666666664,
 'Std of number parmameters of AWQ model': 264.53040152945243,
 'AWQ model size': 1781045.3333333333,
 'Std of AWQ_model_size': 8464.972848942478,
 'AWQ finetune model accuracy': 0.903,
 'Std of AWQ finetune model accuracy': 0.011,
 'time inference of AWQ finetune model': 16.423,
 'Std of time inference of AWQ finetune model': 0.136,
 'number parmameters of AWQ finetune model': 69143,
 'Std of number parmameters of AWQ finetune model': 0.0,
 'AWQ finetune model size': 2212576,